# EXP-01 — Fresh local portability gate
Run through scripts/exp01-colab.py local-pack. Real pinned pilot only; no synthetic fallback. Not cloud acceptance.

In [ ]:
import os, sys
from pathlib import Path
root = Path.cwd().resolve()
sys.dont_write_bytecode = True
sys.path.insert(0, str(root / 'notebooks'))
import colab_runtime as cr
work = Path(os.environ['EXP01_LOCAL_WORK'])
env = cr.environment(root)
identity = cr.source_identity(root)
bundle = cr.pilot(root, Path(os.environ['EXP01_PILOT_ZIP']), work, os.environ['EXP01_RECIPIENT'])
print('REAL_PILOT_VERIFIED', bundle['features_table'].num_rows, bundle['mainshocks_table'].num_rows)


In [ ]:
from baseline_models import generate_local_smoke_report
results = cr.baselines(root, bundle, work / 'baseline', identity)
report = generate_local_smoke_report(bundle, results, work / 'local-smoke-report.json', code_version='source-' + identity['sha256'])
assert report['runner_mode'] == 'nbclient' and report['fresh_kernel_acceptance'] is True
assert report['local_smoke_status'] == 'PASSED' and report['colab_migration_allowed'] is True
for algorithm in cr.MODELS:
    cr.verify_result(work / 'baseline' / algorithm.lower(), algorithm, identity['sha256'])
print('LOCAL_REAL_PILOT_FRESH_KERNEL_PASSED', report['pilot_id'])
